# Mental Health Care Access in Almere

*Daniel Hoog*

This notebook looks at where outpatient mental health care (GGZ) is located in Almere, how that matches the neighbourhoods where the need for it seems highest, and where a new location would make the biggest difference. Everything is built on public data: RIVM, CBS, PDOK and OpenStreetMap, plus a list of care providers I put together and checked by hand.

**How to run:** the first run downloads the raw data from the APIs and stores it in `data/`. After that, every load cell reads from the local copy, so *Restart & Run All* works offline and doesn't hit the APIs again. The neighbourhood boundaries (`wijkenbuurten_2024.gpkg`) have to be downloaded once by hand, see section 5.2.

## 1. Problem Definition

**Mental health problems are common in the Netherlands, and they are becoming more common.** The most recent national population study, NEMESIS-3 (2019–2022), found that 25.9% of people aged 18 to 75 met the criteria for at least one mental disorder in the past year [1]. Compared with 2007–2009, the 12-month prevalence of anxiety and mood disorders rose by 56% to 125%, depending on the disorder [2]. The increase was strongest among students, young adults (18–34) and people living in cities, and it was not explained by the COVID-19 pandemic [3].

**Demand is outrunning supply.** In 2025, people ended up on a mental health waiting list more than 100,000 times. Almost half of them waited longer than the agreed maximum of 14 weeks, and the average wait before treatment started was 24 weeks [4].

**Almere scores worse than the national average on most mental health indicators.** The 2022 GGD Health Monitor for adults (18–64) shows [5]:

| Indicator (adults 18–64, 2022) | Almere | Netherlands |
|---|---|---|
| Severely lonely | 21% | 15% |
| High risk of anxiety disorder or depression | 14% | 11% |
| (Very) much stress in the past 4 weeks | 32% | 25% |
| Moderate or severe psychological complaints | 13% | 9% |
| Serious suicidal thoughts in the past 12 months | 15% | 12% |
| Difficulty making ends meet in the past 12 months | 29% | 21% |

*Source: GGD Flevoland, Gezondheidsmonitor Volwassenen en Ouderen 2022 [5]. Whether each difference is statistically significant is marked in the source table.*

These are city-wide averages. They say nothing about differences *within* Almere, and that is the level this analysis works at.

**Almere is still growing.** The city went from empty polder to roughly 230,000 inhabitants in about fifty years and is expected to keep growing [6]. Political parties have argued for 25,000 new homes in Almere Pampus, enough for 50,000 to 60,000 people [7]. Care facilities will have to keep up.

**Location affects who gets care.** Research on a Swiss mental health system found a clear *distance decay effect*: the longer the public transport travel time to the nearest facility, the less people used outpatient services. For inpatient care this effect was largely absent [8]. The authors' advice was to place outpatient clinics decentrally, close to where people live. So where outpatient care is located has a direct effect on who ends up receiving it.

## 2. Research Question

> **Which neighbourhoods in Almere combine a high need for mental health care with poor access to existing outpatient facilities, and where would a new facility add the most value?**

Sub-questions:

1. How are the existing outpatient mental health facilities spread across Almere?
2. Which neighbourhoods show the highest mental health need (anxiety/depression risk, stress, loneliness, financial strain, suicidal thoughts)?
3. How accessible is the nearest facility from each neighbourhood?
4. Which neighbourhoods score highest when need and access are combined, and how sensitive is that ranking to the choices behind the score?

## 3. Scope

- **Area:** municipality of Almere, at neighbourhood (*buurt*) level
- **Care:** outpatient mental health care for adults, since that is where distance matters most. Youth care is kept in the data but analysed separately
- **Data:** public sources only (RIVM, CBS, GGD Flevoland, PDOK, OpenStreetMap, ZorgkaartNederland)
- **Nature of the analysis:** exploratory and meant to support decisions. It is not a causal study and it does not replace the knowledge of local care providers or the municipality

## References

1. RIVM – Mentale Gezondheid Monitor, *Enige psychische aandoening* (NEMESIS-3). [Link](https://www.rivm.nl/mentale-gezondheid/monitor/algemene-bevolking/enige-psychische-aandoening)
2. *Anxiety and mood disorders on the rise: exploring clinical profiles and risk factors in the Netherlands*, European Psychiatry. [Link](https://resolve.cambridge.org/core/journals/european-psychiatry/article/anxiety-and-mood-disorders-on-the-rise-exploring-clinical-profiles-and-risk-factors-in-the-netherlands/E7C962D5DA4E8CDE5F2AE047C5A8B051)
3. *Prevalence and trends of common mental disorders from 2007–2009 to 2019–2022: results from NEMESIS*. [Link](https://pure.eur.nl/en/publications/prevalence-and-trends-of-common-mental-disorders-from-2007-2009-t/)
4. Radar (AVROTROS) & MIND – *Wachttijden bij de ggz lopen op tot twee jaar*. [Link](https://radar.avrotros.nl/artikel/wachttijden-ggz-lopen-op-tot-twee-jaar-62372)
5. GGD Flevoland – *Gezondheidsmonitor Volwassenen en Ouderen 2022, alle gemeenten (volwassenen)*. [Link](https://ggdflevoland.nl/app/uploads/sites/6/2024/12/Gezondheidsmonitor-Volwassenen-en-Ouderen-2022-Gemeenten-Volwassenen-versie1.pdf)
6. Gemeente Almere – *Ontwerp Omgevingsvisie Almere 2050* (2025). [Link](https://pas.commissiemer.nl/files/nl/3768/08c-bl-ontwerp-omgevingsvisie-almere-2050-ontwerp-omgevingsvisie.pdf)
7. Binnenlands Bestuur – *Coalitiepartijen: 25.000 woningen in Almere*. [Link](https://www.binnenlandsbestuur.nl/ruimte-en-milieu/privacy/coalitiepartijen-25-000-woningen-in-almere)
8. Stulz, N., Pichler, E.-M., Kawohl, W., & Hepp, U. (2018). *The gravitational force of mental health services: distance decay effects in a rural Swiss service area*. BMC Health Services Research. [Link](https://pmc.ncbi.nlm.nih.gov/articles/PMC5800291)

## 4. Setup

In [ ]:
import os
import time

import pandas as pd
import numpy as np
import cbsodata
import requests
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)

GM_ALMERE = "GM0034"  # CBS municipality code
BU_ALMERE = "BU0034"  # prefix of every neighbourhood code in Almere

os.makedirs("data", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Some APIs refuse requests without a User-Agent (Overpass gave a 406)
HEADERS = {
    "User-Agent": "almere-mental-health-access/1.0 (portfolio project; contact@optivueconsultancy.nl)",
    "Accept": "application/json",
}

## 5. Data

| Source | What I use it for | Level |
|---|---|---|
| RIVM *Gezondheid per wijk en buurt* (CBS table 50150NED) | Mental health indicators (model estimates) | Neighbourhood |
| CBS *Kerncijfers wijken en buurten 2024* (85984NED) | Population, households, income, welfare | Neighbourhood |
| PDOK *Wijk- en buurtkaart 2024* | Neighbourhood boundaries | Neighbourhood |
| OpenStreetMap (Overpass API) | Care locations (first attempt), public transport stops | Point |
| ZorgkaartNederland | Checking which providers actually have a location in Almere | Point |
| PDOK Locatieserver | Turning addresses into coordinates | Point |

The RIVM figures are small-area estimates: a model combines the GGD Health Monitor with registry data to predict values per neighbourhood. They are good for comparing neighbourhoods, but they are not measurements.

### 5.1 RIVM and CBS tables

Both tables come from the CBS OData API. The RIVM table sits in the third-party catalogue (`dataderden.cbs.nl`). The download is large (the RIVM table alone is over 800,000 rows), so it is stored as parquet after the first run.

In [ ]:
def load_cbs_table(table_id, path, catalog_url=None):
    """Load a CBS table from the local cache, or download it once."""
    if os.path.exists(path):
        return pd.read_parquet(path)
    kwargs = {"catalog_url": catalog_url} if catalog_url else {}
    df = pd.DataFrame(cbsodata.get_data(table_id, **kwargs))
    df.to_parquet(path)
    return df


rivm_raw = load_cbs_table("50150NED", "data/rivm_raw.parquet", catalog_url="dataderden.cbs.nl")
kern_raw = load_cbs_table("85984NED", "data/kern_raw.parquet")

print("RIVM:", rivm_raw.shape)
print("CBS: ", kern_raw.shape)

### 5.2 Neighbourhood boundaries

The *Wijk- en buurtkaart 2024* GeoPackage is downloaded by hand from PDOK and saved as `data/wijkenbuurten_2024.gpkg`. It has three layers (`gemeenten`, `wijken`, `buurten`) in the Dutch RD New projection (EPSG:28992), which works in metres and is handy for distances later on.

I only keep land neighbourhoods. The water polygons (`water == "NEE"` means *not* water) would otherwise show up as empty areas in the maps.

In [ ]:
buurten_raw = gpd.read_file("data/wijkenbuurten_2024.gpkg", layer="buurten")

buurten = buurten_raw[
    (buurten_raw["gemeentecode"] == GM_ALMERE)
    & (buurten_raw["water"] == "NEE")
][["buurtcode", "buurtnaam", "geometry"]].copy()

print(buurten.crs)
print(f"Neighbourhoods in Almere (land): {len(buurten)}")

### 5.3 OpenStreetMap: care locations and public transport

My first idea was to pull the mental health care locations straight from OpenStreetMap. The query below searches on the relevant healthcare tags and also on names containing *ggz*, *psycholo*, *psychiat* or *psychotherap*. The same API gives the bus stops and train stations, which I'll use later for public transport access.

The public Overpass servers are often busy, so there is a backup server and the results are cached.

In [ ]:
OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",  # backup
]


def run_overpass(query):
    """Run an Overpass query, falling back to the next server on failure."""
    for url in OVERPASS_URLS:
        try:
            resp = requests.post(url, data={"data": query}, headers=HEADERS, timeout=90)
            resp.raise_for_status()
            return resp.json()["elements"]
        except requests.RequestException as e:
            print(f"Failed on {url}: {e}")
    raise RuntimeError("All Overpass servers failed.")


def elements_to_df(elements):
    """Flatten Overpass elements into a DataFrame."""
    return pd.DataFrame([{
        "osm_id": e["id"],
        "osm_type": e["type"],
        "name": e.get("tags", {}).get("name"),
        "healthcare": e.get("tags", {}).get("healthcare"),
        "speciality": e.get("tags", {}).get("healthcare:speciality"),
        # nodes have lat/lon, ways and relations have a center
        "lat": e.get("lat", e.get("center", {}).get("lat")),
        "lon": e.get("lon", e.get("center", {}).get("lon")),
    } for e in elements])


def load_overpass(query, path):
    """Load Overpass results from cache, or query once."""
    if os.path.exists(path):
        return pd.read_parquet(path)
    try:
        df = elements_to_df(run_overpass(query))
    except RuntimeError as e:
        print(f"{e} Try again later; continuing without this data.")
        return pd.DataFrame(columns=["osm_id", "osm_type", "name", "healthcare", "speciality", "lat", "lon"])
    df.to_parquet(path)
    return df

In [ ]:
# admin_level 8 = municipality
ggz_query = """
[out:json][timeout:60];
area["name"="Almere"]["admin_level"="8"]->.a;
(
  nwr["healthcare"~"psychotherapist|counselling|psychologist"](area.a);
  nwr["healthcare:speciality"~"psychiatry|psychology|psychotherapy"](area.a);
  nwr["name"~"ggz|psycholo|psychiat|psychotherap",i](area.a);
);
out center tags;
"""

ov_query = """
[out:json][timeout:60];
area["name"="Almere"]["admin_level"="8"]->.a;
(
  nwr["highway"="bus_stop"](area.a);
  nwr["railway"="station"](area.a);
);
out center tags;
"""

ggz_osm = load_overpass(ggz_query, "data/ggz_osm.parquet")
ov_raw = load_overpass(ov_query, "data/ov_raw.parquet")

print(f"Care locations in OSM: {len(ggz_osm)}")
print(f"Public transport stops: {len(ov_raw)}")
ggz_osm[["name", "healthcare", "speciality"]]

### 5.4 Why OSM is not enough: a verified provider list

OSM returns only five locations, and two of them don't belong in this analysis (a forensic psychiatric centre and a youth/family centre). Large providers such as GGz Centraal are missing entirely. Using this as the supply side would make almost every neighbourhood look underserved.

So I built the list myself in `data/ggz_manual.csv`. I went through the mental health providers listed for Almere on ZorgkaartNederland and checked each location on the provider's own website. Every row has a source link.

**Inclusion criteria** for the adult analysis:

- Offers outpatient (ambulant) treatment
- Accessible to the general public through a GP referral
- Has a physical location in Almere

**Excluded:** forensic care, inpatient-only or sheltered housing, services for one specific target group (e.g. autism only), and organisations without a treatment location (care delivered at home).

The `include` column is `TRUE`, `FALSE` or `youth`. Youth providers are kept in the file because they are part of the picture, but the main analysis is about adults, since the RIVM indicators are for adults too.

### 5.5 Geocoding the provider list

The list only has addresses. The PDOK Locatieserver turns them into coordinates. It is the government's address service, built on the national address register (BAG), so for Dutch addresses it is more reliable than a general geocoder.

In [ ]:
PDOK_URL = "https://api.pdok.nl/bzk/locatieserver/search/v3_1/free"


def geocode_pdok(address):
    """Return (lat, lon) for an address, or (None, None) if not found."""
    params = {"q": address, "fq": "type:adres", "rows": 1}
    resp = requests.get(PDOK_URL, params=params, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    docs = resp.json()["response"]["docs"]
    if not docs:
        return None, None
    # format: 'POINT(lon lat)'
    lon, lat = docs[0]["centroide_ll"].replace("POINT(", "").replace(")", "").split()
    return float(lat), float(lon)


if os.path.exists("data/ggz_locations.gpkg"):
    ggz_gdf = gpd.read_file("data/ggz_locations.gpkg")
else:
    ggz = pd.read_csv("data/ggz_manual.csv")
    coords = []
    for address in ggz["address"]:
        coords.append(geocode_pdok(address))
        time.sleep(0.2)  # be polite to the free API
    ggz[["lat", "lon"]] = pd.DataFrame(coords, index=ggz.index)

    ggz_gdf = gpd.GeoDataFrame(
        ggz, geometry=gpd.points_from_xy(ggz["lon"], ggz["lat"]), crs="EPSG:4326"
    )
    ggz_gdf.to_file("data/ggz_locations.gpkg", driver="GPKG")

print(f"Geocoded: {ggz_gdf['lat'].notna().sum()} / {len(ggz_gdf)}")
ggz_gdf[["name", "target_group", "include", "lat", "lon"]]

All 19 addresses were found. 11 locations qualify for the adult analysis, 5 are youth care and 3 are excluded.

## 6. Building the neighbourhood dataset

### 6.1 Mental health need (RIVM)

From the RIVM table I keep Almere's neighbourhoods, the most recent year (2024), all adults (18+) and the point estimate rather than the confidence margins. CBS text columns often have trailing spaces, so those are stripped first, otherwise the filters silently return nothing.

The five indicators are the ones closest to mental health need, and they match the GGD table in section 1:

| RIVM column | Name here |
|---|---|
| `HoogRisicoAngststoornisOfDepressie_14` | `high_risk_anxiety_depression` |
| `HeelVeelStressAfg4Weken_17` | `high_stress` |
| `SterkEenzaam_26` | `severe_loneliness` |
| `MoeiteMetRondkomen_32` | `financial_difficulty` |
| `SuicideGedachtenDeLaatste12Maanden_15` | `suicidal_thoughts` |

All values are percentages of adults in the neighbourhood.

In [ ]:
rivm = rivm_raw.copy()
for col in ["Leeftijd", "Marges", "Perioden", "SoortRegio_2", "Codering_3", "WijkenEnBuurten"]:
    rivm[col] = rivm[col].str.strip()

rivm_almere = rivm[
    (rivm["SoortRegio_2"] == "Buurt")
    & rivm["Codering_3"].str.startswith(BU_ALMERE)
    & (rivm["Perioden"] == "2024")
    & (rivm["Leeftijd"] == "18 jaar of ouder")
    & (rivm["Marges"] == "Waarde")  # point estimate, not the margins
]

indicator_map = {
    "Codering_3": "buurtcode",
    "WijkenEnBuurten": "buurt_name",
    "HoogRisicoAngststoornisOfDepressie_14": "high_risk_anxiety_depression",
    "HeelVeelStressAfg4Weken_17": "high_stress",
    "SterkEenzaam_26": "severe_loneliness",
    "MoeiteMetRondkomen_32": "financial_difficulty",
    "SuicideGedachtenDeLaatste12Maanden_15": "suicidal_thoughts",
}
need = rivm_almere[list(indicator_map)].rename(columns=indicator_map)

print(need.shape)
need.isna().sum()

### 6.2 Neighbourhood context (CBS)

From the CBS key figures I take population and a few background variables. Counts are turned into shares so that large and small neighbourhoods can be compared. These variables are not part of the score yet, but they help explain it and are needed for the check below.

In [ ]:
kern = kern_raw.copy()
kern["Codering_3"] = kern["Codering_3"].str.strip()
kern_almere = kern[kern["Codering_3"].str.startswith(BU_ALMERE)]

context_map = {
    "Codering_3": "buurtcode",
    "AantalInwoners_5": "population",
    "HuishoudensTotaal_29": "households_total",
    "Eenpersoonshuishoudens_30": "single_person_hh",
    "GemiddeldInkomenPerInwoner_78": "income_per_resident",  # x 1,000 euro
    "PersonenPerSoortUitkeringBijstand_87": "welfare_recipients",
    "AfstandTotHuisartsenpraktijk_110": "distance_to_gp_km",
    "MateVanStedelijkheid_120": "urbanity",  # 1 = very urban, 5 = not urban
}
context = kern_almere[list(context_map)].rename(columns=context_map)

context["share_single_person_hh"] = context["single_person_hh"] / context["households_total"] * 100
context["welfare_per_1000"] = context["welfare_recipients"] / context["population"] * 1000

print(context.shape)

### 6.3 Neighbourhoods without an estimate

74 of the 238 neighbourhoods have no RIVM estimate at all. Before dropping them I want to know who lives there: if these were populated residential areas, leaving them out would bias the results.

In [ ]:
no_estimate = need[need["high_risk_anxiety_depression"].isna()]
check = no_estimate.merge(context[["buurtcode", "population"]], on="buurtcode", how="left")

total_pop = context["population"].sum()
missing_pop = check["population"].sum()

print(f"Neighbourhoods without estimate: {len(check)}")
print(f"Residents living there: {missing_pop:,.0f} of {total_pop:,.0f} ({missing_pop / total_pop:.2%})")
print(f"Median residents per neighbourhood: {check['population'].median():.0f}")

They house about 1,030 of Almere's ~226,000 residents (0.45%), and the median is 0 residents. These are business parks, parks, woodland and other areas where hardly anyone lives. RIVM doesn't publish estimates for them, and dropping them doesn't affect the analysis. That leaves 164 neighbourhoods.

In [ ]:
need_valid = need.dropna(subset=["high_risk_anxiety_depression"]).copy()
print(f"Neighbourhoods with estimates: {len(need_valid)}")

### 6.4 One table with geometry

Boundaries, need indicators and context are joined on `buurtcode`. The inner join with the RIVM data keeps only the 164 neighbourhoods with an estimate.

In [ ]:
gdf = (
    buurten
    .merge(need_valid, on="buurtcode", how="inner")
    .merge(context, on="buurtcode", how="left")
)

print(gdf.shape)
print(f"Duplicate neighbourhoods: {gdf['buurtcode'].duplicated().sum()}")
gdf.head()

## 7. Access: distance to the nearest outpatient facility

For every neighbourhood I measure the straight-line distance to the nearest adult outpatient location. Both layers are in EPSG:28992, so the result is in metres.

As the starting point of each neighbourhood I use `representative_point()` instead of the centroid. For oddly shaped areas the centroid can fall outside the polygon (in a lake, say); a representative point is always inside.

In [ ]:
ggz_adult = ggz_gdf[ggz_gdf["include"].astype(str).str.upper() == "TRUE"].to_crs(epsg=28992)
print(f"Adult outpatient locations: {len(ggz_adult)}")

points = gdf[["buurtcode", "geometry"]].copy()
points["geometry"] = points.geometry.representative_point()

nearest = gpd.sjoin_nearest(
    points, ggz_adult[["name", "geometry"]], how="left", distance_col="dist_to_ggz_m"
).drop_duplicates("buurtcode")  # ties would give two rows

gdf = gdf.merge(
    nearest[["buurtcode", "name", "dist_to_ggz_m"]].rename(columns={"name": "nearest_ggz"}),
    on="buurtcode",
    how="left",
)
gdf["dist_to_ggz_km"] = gdf["dist_to_ggz_m"] / 1000

gdf["dist_to_ggz_km"].describe().round(2)

Half of the neighbourhoods are within about 1.2 km of an outpatient location, and three quarters within 2.2 km. The tail is long, though: the furthest neighbourhood is 7 km away.

**Limitation:** straight-line distance is not travel time. A canal, a motorway or a poor bus connection can make a short distance a long trip. Public transport access is the next step (see the end of this notebook).

## 8. Priority score

The score combines two parts:

- **Need score:** each of the five indicators is scaled to 0–1 across Almere's neighbourhoods (0 = lowest in Almere, 1 = highest), then averaged. All five carry equal weight.
- **Access gap:** the distance to the nearest facility, scaled to 0–1 the same way.

`priority_score = 0.5 × need_score + 0.5 × access_gap`

The scores are relative: a 1 means "highest within Almere", not "high" in any absolute sense. The 50/50 split is a starting choice, not a fact. Section 4 of the research question is about testing how much the ranking moves when the weights change.

In [ ]:
need_cols = [
    "high_risk_anxiety_depression",
    "high_stress",
    "severe_loneliness",
    "financial_difficulty",
    "suicidal_thoughts",
]

need_scaled = MinMaxScaler().fit_transform(gdf[need_cols])
gdf["need_score"] = need_scaled.mean(axis=1)

gdf["access_gap"] = MinMaxScaler().fit_transform(gdf[["dist_to_ggz_km"]]).ravel()

gdf["priority_score"] = 0.5 * gdf["need_score"] + 0.5 * gdf["access_gap"]

top10 = gdf.nlargest(10, "priority_score")[
    ["buurtnaam", "population", "need_score", "dist_to_ggz_km", "nearest_ggz", "priority_score"]
]
top10.round(2)

Keep the `population` column in mind when reading this table. The score treats every neighbourhood the same, regardless of how many people live there. A sparsely populated area on the edge of the city can end up high mostly because it is far away. Whether that is a real gap or an artefact is something the next step has to settle.

## 9. Maps

### 9.1 Static map

A static version, so the result is also visible on GitHub, which doesn't render interactive maps.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

gdf.plot(
    column="priority_score",
    cmap="YlOrRd",
    edgecolor="white",
    linewidth=0.3,
    legend=True,
    legend_kwds={"label": "Priority score", "shrink": 0.6},
    ax=ax,
)
ggz_adult.plot(ax=ax, color="#1f4e8c", markersize=40, edgecolor="white", label="Outpatient GGZ location")

ax.set_title("Mental health care priority by neighbourhood, Almere", fontsize=14)
ax.legend(loc="lower left")
ax.set_axis_off()

plt.savefig("output/map_priority_score.png", dpi=150, bbox_inches="tight")
plt.show()

### 9.2 Interactive map

Hover over a neighbourhood to see its numbers. Plotly needs plain longitude/latitude (EPSG:4326), and the boundaries are simplified a little (10 m tolerance) so the notebook stays light.

In [ ]:
gdf_web = gdf.copy()
gdf_web["geometry"] = gdf_web.geometry.simplify(10)
gdf_web = gdf_web.to_crs(epsg=4326).set_index("buurtcode")

ggz_web = ggz_adult.to_crs(epsg=4326)

fig = px.choropleth_map(
    gdf_web,
    geojson=gdf_web.geometry,
    locations=gdf_web.index,
    color="priority_score",
    color_continuous_scale="YlOrRd",
    hover_name="buurtnaam",
    hover_data={
        "population": True,
        "need_score": ":.2f",
        "dist_to_ggz_km": ":.2f",
        "nearest_ggz": True,
        "priority_score": ":.2f",
    },
    map_style="carto-positron",
    center={"lat": 52.37, "lon": 5.22},
    zoom=10.5,
    opacity=0.7,
)

fig.add_scattermap(
    lat=ggz_web.geometry.y,
    lon=ggz_web.geometry.x,
    mode="markers",
    marker={"size": 10, "color": "#1f4e8c"},
    text=ggz_web["name"],
    hoverinfo="text",
    name="Outpatient GGZ location",
)

fig.update_layout(height=650, margin={"l": 0, "r": 0, "t": 0, "b": 0})
fig.show()

## Next steps

- **Small neighbourhoods:** check how many of the top 10 have very few residents, and decide whether to set a minimum population or weight by population
- **Sensitivity analysis:** vary the need/access weights (e.g. 70/30, 30/70) and see which neighbourhoods stay at the top
- **Public transport:** use the bus stops and stations from section 5.3 to add a second access measure next to straight-line distance
- **Conclusions and limitations:** answer the research question and name what this analysis can't tell us